<a href="https://colab.research.google.com/github/strongman2026/colab/blob/main/comfyui_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1：基础环境与网盘挂载（开机跑一次）

In [1]:
import os, sys, subprocess
from google.colab import drive

print("☁️ 1. 正在挂载 Google Drive (弹窗请点击允许)...")
drive.mount('/content/drive')

COMFY_DIR = "/content/ComfyUI"
FRP_DIR = "/content/frp"
FRPC_BIN = f"{FRP_DIR}/frpc"

# ================= 🚨 必填配置 =================
VPS_IP = "34.153.197.18"  # 你的服务器 IP
FRP_TOKEN = "Kaggle2026!" # 你的服务器密码
# ===============================================

print("📥 2. 部署 ComfyUI 核心与管理器...")
if not os.path.exists(COMFY_DIR):
    os.system(f"git clone https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR} -q")
    os.system(f"git clone https://github.com/ltdrdata/ComfyUI-Manager.git {COMFY_DIR}/custom_nodes/ComfyUI-Manager -q")

print("📦 3. 安装环境依赖...")
if os.path.exists(f"{COMFY_DIR}/requirements.txt"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", f"{COMFY_DIR}/requirements.txt", "-q"])

print("🔧 4. 部署 FRP 客户端与配置文件...")
if not os.path.exists(FRPC_BIN):
    os.makedirs(FRP_DIR, exist_ok=True)
    os.system(f"wget -q -O {FRP_DIR}/frp.tar.gz https://github.com/fatedier/frp/releases/download/v0.61.1/frp_0.61.1_linux_amd64.tar.gz")
    os.system(f"tar -xzf {FRP_DIR}/frp.tar.gz -C {FRP_DIR} --strip-components=1")
    os.system(f"chmod +x {FRPC_BIN}")

frpc_config = f"""
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.method = "token"
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8188
"""
with open(f"{FRP_DIR}/frpc.toml", "w") as f:
    f.write(frpc_config)

print("✅ Cell 1 完成！基础环境已就绪。")

☁️ 1. 正在挂载 Google Drive (弹窗请点击允许)...
Mounted at /content/drive
📥 2. 部署 ComfyUI 核心与管理器...
📦 3. 安装环境依赖...
🔧 4. 部署 FRP 客户端与配置文件...
✅ Cell 1 完成！基础环境已就绪。


Cell 2

In [2]:
import os

COMFY_DIR = "/kaggle/working/ComfyUI"
MODELS_DIR = f"{COMFY_DIR}/models"
OUTPUT_DIR = f"{COMFY_DIR}/output"

# =================================================================
# 1️⃣ Kaggle 原生数据集（右侧 Data 导入的，0秒挂载，极致读取速度）
# =================================================================
KAGGLE_DATASETS = [
    ("checkpoints", "/kaggle/input/animagine-xl-4-0-comfyui*/*.safetensors"),
    ("checkpoints", "/kaggle/input/datasets/jarredstroman/my-pony-v6-private/*.safetensors"),
    ("loras", "/kaggle/input/datasets/jarredstroman/family-guy-style/*.safetensors"),
]

# =================================================================
# 2️⃣ 外部直链极速下载（用 aria2c 从 C站或 HuggingFace 拉取）
# =================================================================
EXTERNAL_URLS = [
    # 格式: ("文件夹名", "模型直链")
    # ("checkpoints", "https://huggingface.co/.../xxx.safetensors"),
]

print("🧹 1. 清理旧挂载与旧缓存...")
os.system(f"find {MODELS_DIR}/checkpoints -type l -delete")
os.system(f"find {MODELS_DIR}/loras -type l -delete")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("🔗 2. 正在秒级挂载 Kaggle 本地数据集模型...")
for folder, path in KAGGLE_DATASETS:
    os.makedirs(f"{MODELS_DIR}/{folder}", exist_ok=True)
    os.system(f"ln -sf {path} {MODELS_DIR}/{folder}/")
    print(f"  -> 已挂载 [{folder}]: {path.split('/')[-2]}")

if EXTERNAL_URLS:
    print("\n🚀 3. 部署多线程下载神器 aria2...")
    os.system("apt-get -y install -qq aria2")

    print("📥 4. 正在以最高百兆/秒拉取外部直链模型到本地...")
    for folder, url in EXTERNAL_URLS:
        dest_dir = f"{MODELS_DIR}/{folder}"
        os.makedirs(dest_dir, exist_ok=True)
        print(f"⚡ 极限拉取中: {url.split('/')[-1]}")
        # 16线程并发下载，榨干 Kaggle 万兆带宽
        os.system(f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M '{url}' -d '{dest_dir}'")

print("\n🎉 模型部署完毕！全部基于 Kaggle NVMe 高速本地硬盘！")
print(f"🖼️ 你的图片将极速生成至: {OUTPUT_DIR}")

🧹 1. 清理旧挂载与旧缓存...
🔗 2. 正在秒级挂载 Kaggle 本地数据集模型...
  -> 已挂载 [checkpoints]: animagine-xl-4-0-comfyui*
  -> 已挂载 [checkpoints]: my-pony-v6-private
  -> 已挂载 [loras]: family-guy-style

🎉 模型部署完毕！全部基于 Kaggle NVMe 高速本地硬盘！
🖼️ 你的图片将极速生成至: /kaggle/working/ComfyUI/output


In [5]:
import os
from google.colab import userdata

# 1. 从 Colab Secrets 安全读取 HF Token
try:
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("✅ 成功从 Secrets 读取 HF_TOKEN！")
except userdata.SecretNotFoundError:
    print("❌ 找不到 Token！请点击左侧 🔑 图标添加 HF_TOKEN，并允许 Notebook 访问。")

# 2. 安装多线程下载神器 aria2
!apt -y install -qq aria2

# 3. 确保目录存在
!mkdir -p /content/ComfyUI/models/vae/
!mkdir -p /content/ComfyUI/models/clip/

# 4. 极速下载（带上 Token 鉴权头，彻底防拦截）
print("🚀 开始极速下载 VAE (ae.safetensors)...")
!aria2c --header="Authorization: Bearer $HF_TOKEN" --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors -d /content/ComfyUI/models/vae -o ae.safetensors

print("🚀 开始极速下载 CLIP-L (clip_l.safetensors)...")
!aria2c --header="Authorization: Bearer $HF_TOKEN" --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors -d /content/ComfyUI/models/clip -o clip_l.safetensors

print("🚀 开始极速下载满血版 T5XXL (t5xxl_fp16.safetensors，约 9.8GB)...")
!aria2c --header="Authorization: Bearer $HF_TOKEN" --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp16.safetensors -d /content/ComfyUI/models/clip -o t5xxl_fp16.safetensors

print("🎉 所有 Flux 必需的齿轮已带 Token 鉴权安全下载完毕！")

✅ 成功从 Secrets 读取 HF_TOKEN！
aria2 is already the newest version (1.36.0-1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
🚀 开始极速下载 VAE (ae.safetensors)...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
c8ce20|OK  |       0B/s|/content/ComfyUI/models/vae/ae.safetensors

Status Legend:
(OK):download completed.
🚀 开始极速下载 CLIP-L (clip_l.safetensors)...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
23a349|OK  |       0B/s|/content/ComfyUI/models/clip/clip_l.safetensors

Status Legend:
(OK):download completed.
🚀 开始极速下载满血版 T5XXL (t5xxl_fp16.safetensors，约 9.8GB)...

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
812d2f|OK  |       0B/s|/content/ComfyUI/models/clip/t5xxl_fp16.safetensors

Status Legend:
(OK):download completed.


In [6]:
import os
from google.colab import userdata

# 1. 从 Colab Secrets 安全读取 Kaggle Token
try:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')
    print("✅ 成功从 Colab Secrets 读取 API Token！")
except userdata.SecretNotFoundError:
    print("❌ 找不到 Token！请点击左侧 🔑 图标添加名为 KAGGLE_API_TOKEN 的密钥，并允许 Notebook 访问。")

# 2. 升级 Kaggle CLI 确保支持新版 Token
!pip install -q -U kaggle

# 3. 创建接收目录并下载 (利用 Colab 神仙网速拉取 23GB 模型)
!mkdir -p /content/dataset
print("⏳ 正在从 Kaggle 拉取 23GB 的 Flux 模型...")
!kaggle datasets download -d jarredstroman/flux-2d-assets-2026-v12 -p /content/dataset --unzip

# 4. 建立软链接到 ComfyUI 目录
!mkdir -p /content/ComfyUI/models/checkpoints/
!ln -s /content/dataset/flux1-schnell.safetensors /content/ComfyUI/models/checkpoints/flux1-schnell.safetensors

print("✅ 模型已成功挂载到 ComfyUI！")

✅ 成功从 Colab Secrets 读取 API Token！
⏳ 正在从 Kaggle 拉取 23GB 的 Flux 模型...
Dataset URL: https://www.kaggle.com/datasets/jarredstroman/flux-2d-assets-2026-v12
License(s): CC0-1.0
100% 17.6G/17.6G [02:14<00:00, 141MB/s]

✅ 模型已成功挂载到 ComfyUI！


In [10]:
import os
from google.colab import userdata

# 1. 从 Colab Secrets 安全读取 Kaggle Token
try:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get('KAGGLE_API_TOKEN')
    print("✅ 成功从 Colab Secrets 读取 API Token！")
except userdata.SecretNotFoundError:
    print("❌ 找不到 Token！请点击左侧 🔑 图标添加名为 KAGGLE_API_TOKEN 的密钥，并允许 Notebook 访问。")

# 2. 升级 Kaggle CLI 确保支持新版 Token
!pip install -q -U kaggle

# 3. 创建接收目录并下载
!mkdir -p /content/dataset
print("⏳ 正在从 Kaggle 拉取 Flux 模型...")
!kaggle datasets download -d jarredstroman/flux-2d-assets-2026-v12 -p /content/dataset --unzip

# 4. 创建 ComfyUI 模型目录
!mkdir -p /content/ComfyUI/models/unet/
!mkdir -p /content/ComfyUI/models/checkpoints/


# 5. 建立软链接到 ComfyUI 目录
!ln -sf /content/dataset/flux1-schnell.safetensors /content/ComfyUI/models/unet/flux1-schnell.safetensors
!ln -s /content/dataset/flux1-schnell.safetensors /content/ComfyUI/models/checkpoints/flux1-schnell.safetensors

print("✅ 模型已成功挂载到 ComfyUI！")

✅ 成功从 Colab Secrets 读取 API Token！
⏳ 正在从 Kaggle 拉取 Flux 模型...
Dataset URL: https://www.kaggle.com/datasets/jarredstroman/flux-2d-assets-2026-v12
License(s): CC0-1.0
100% 17.6G/17.6G [01:13<00:00, 256MB/s]

ln: failed to create symbolic link '/content/ComfyUI/models/checkpoints/flux1-schnell.safetensors': File exists
✅ 模型已成功挂载到 ComfyUI！


In [13]:
# ============================================================
# ComfyUI 修正版：整理已下载模型 + 安装 IPAdapter/ControlNet 节点
# 适用前提：
#   1) 你已经运行过：
#      - Hugging Face 下载 ae.safetensors / clip_l.safetensors / t5xxl_fp16.safetensors
#      - Kaggle 下载 flux1-schnell.safetensors 到 /content/dataset/
#   2) 现在需要把文件整理到 ComfyUI 正确目录
#   3) 安装 IPAdapter / ControlNet 自定义节点
# ============================================================

import os
from google.colab import userdata

# ----------------------------
# 1) 安装基础依赖
# ----------------------------
!pip install -q -U huggingface_hub

# ----------------------------
# 2) 安装自定义节点
# ----------------------------
%cd /content/ComfyUI/custom_nodes

# IPAdapter 节点
!if [ ! -d "ComfyUI_IPAdapter_plus" ]; then git clone https://github.com/cubiq/ComfyUI_IPAdapter_plus.git; fi

# ControlNet 辅助预处理节点
!if [ ! -d "comfyui_controlnet_aux" ]; then git clone https://github.com/Fannovel16/comfyui_controlnet_aux.git; fi

# 可选工具节点
!if [ ! -d "rgthree-comfy" ]; then git clone https://github.com/rgthree/rgthree-comfy.git; fi

# 安装依赖（存在就装，不存在就跳过）
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus/requirements.txt || true
!pip install -q -r /content/ComfyUI/custom_nodes/comfyui_controlnet_aux/requirements.txt || true
!pip install -q -r /content/ComfyUI/custom_nodes/rgthree-comfy/requirements.txt || true

# ----------------------------
# 3) 创建标准模型目录
# ----------------------------
# 注意：Flux 主���型不要放 checkpoints，这里统一放到正确目录
!mkdir -p /content/ComfyUI/models/unet
!mkdir -p /content/ComfyUI/models/clip
!mkdir -p /content/ComfyUI/models/vae
!mkdir -p /content/ComfyUI/models/ipadapter
!mkdir -p /content/ComfyUI/models/clip_vision
!mkdir -p /content/ComfyUI/models/controlnet
!mkdir -p /content/dataset

# ----------------------------
# 4) 工具函数：安全删除旧链接/旧文件并创建新软链接
# ----------------------------
def safe_symlink(src, dst, desc):
    try:
        if os.path.islink(dst) or os.path.exists(dst):
            os.remove(dst)
        if os.path.exists(src):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            os.symlink(src, dst)
            print(f"✅ {desc}: {src} -> {dst}")
        else:
            print(f"⚠️ 未找到 {desc} 源文件：{src}")
    except Exception as e:
        print(f"❌ {desc} 处理失败：{e}")

# ----------------------------
# 5) 整理你已经下载好的 Flux 文件到正确目录
# ----------------------------
# 你之前 Kaggle 下载的 flux1-schnell.safetensors 大概率在 /content/dataset/
# 这里把它链接到 models/unet/，而不是 models/checkpoints/
safe_symlink(
    "/content/dataset/flux1-schnell.safetensors",
    "/content/ComfyUI/models/unet/flux1-schnell.safetensors",
    "Flux 主模型 flux1-schnell.safetensors"
)

# 你之前 HF 下载的文本编码器和 VAE，放到正确目录
safe_symlink(
    "/content/ComfyUI/models/vae/ae.safetensors",
    "/content/ComfyUI/models/vae/ae.safetensors",
    "VAE ae.safetensors"
)

safe_symlink(
    "/content/ComfyUI/models/clip/clip_l.safetensors",
    "/content/ComfyUI/models/clip/clip_l.safetensors",
    "CLIP-L clip_l.safetensors"
)

safe_symlink(
    "/content/ComfyUI/models/clip/t5xxl_fp16.safetensors",
    "/content/ComfyUI/models/clip/t5xxl_fp16.safetensors",
    "T5XXL t5xxl_fp16.safetensors"
)

# ----------------------------
# 6) 兼容检查：如果 Flux 主模型还被放在 checkpoints/，给你提示
# ----------------------------
if os.path.exists("/content/ComfyUI/models/checkpoints/flux1-schnell.safetensors"):
    print("⚠️ 检测到 /content/ComfyUI/models/checkpoints/flux1-schnell.safetensors")
    print("   这通常不是 Flux + UNETLoader 的正确位置。建议以后不要从 checkpoints 读取它。")

# ----------------------------
# 7) IPAdapter 权重准备
# ----------------------------
# Flux IPAdapter 参考：
#   https://huggingface.co/XLabs-AI/flux-ip-adapter
# 这里不强行下载，假设你后面手动下载到 /content/dataset/ 后再软链接。
#
# 如果你已经把权重放进 /content/dataset/，把文件名改成你实际的即可：
#
#   /content/dataset/flux-ip-adapter.safetensors
#
safe_symlink(
    "/content/dataset/flux-ip-adapter.safetensors",
    "/content/ComfyUI/models/ipadapter/flux-ip-adapter.safetensors",
    "Flux IPAdapter 权重"
)

# ----------------------------
# 8) CLIP Vision 模型准备
# ----------------------------
# IPAdapter 常常还需要 CLIP Vision 编码器
# 常见文件名示例：
#   CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors
safe_symlink(
    "/content/dataset/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors",
    "/content/ComfyUI/models/clip_vision/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors",
    "CLIP Vision 模型"
)

# ----------------------------
# 9) ControlNet 模型准备（可选）
# ----------------------------
# 如果你之后要用控制网，请把模型放进 /content/dataset 后再软链接
# 下面仅做示例，默认不启用
#
# safe_symlink(
#     "/content/dataset/controlnet_canny.safetensors",
#     "/content/ComfyUI/models/controlnet/controlnet_canny.safetensors",
#     "ControlNet Canny 模型"
# )
#
# safe_symlink(
#     "/content/dataset/controlnet_depth.safetensors",
#     "/content/ComfyUI/models/controlnet/controlnet_depth.safetensors",
#     "ControlNet Depth 模型"
# )

# ----------------------------
# 10) 输出当前目录状态
# ----------------------------
print("\n📁 当前 /content/ComfyUI/models 文件列表：")
!find /content/ComfyUI/models -maxdepth 2 -type f | sort

print("\n✅ 处理完成。")
print("⚠️ 请重启 ComfyUI，再重新加载工作流。")
print("⚠️ 如果你之后下载了 IPAdapter / CLIP Vision / ControlNet 模型，再跑一次此 cell 即可自动挂载。")

/content/ComfyUI/custom_nodes
ERROR: Could not open requirements file: [Errno 2] No such file or directory: '/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus/requirements.txt'
✅ Flux 主模型 flux1-schnell.safetensors: /content/dataset/flux1-schnell.safetensors -> /content/ComfyUI/models/unet/flux1-schnell.safetensors
⚠️ 未找到 VAE ae.safetensors 源文件：/content/ComfyUI/models/vae/ae.safetensors
⚠️ 未找到 CLIP-L clip_l.safetensors 源文件：/content/ComfyUI/models/clip/clip_l.safetensors
⚠️ 未找到 T5XXL t5xxl_fp16.safetensors 源文件：/content/ComfyUI/models/clip/t5xxl_fp16.safetensors
⚠️ 检测到 /content/ComfyUI/models/checkpoints/flux1-schnell.safetensors
   这通常不是 Flux + UNETLoader 的正确位置。建议以后不要从 checkpoints 读取它。
⚠️ 未找到 Flux IPAdapter 权重 源文件：/content/dataset/flux-ip-adapter.safetensors
⚠️ 未找到 CLIP Vision 模型 源文件：/content/dataset/CLIP-ViT-bigG-14-laion2B-39B-b160k.safetensors

📁 当前 /content/ComfyUI/models 文件列表：
/content/ComfyUI/models/audio_encoders/put_audio_encoder_models_here
/content/ComfyUI/models/checkpoints

cell3 极速启动与物理隔离（日常只需跑这个）

In [11]:
import os, time, subprocess, socket

COMFY_DIR = "/content/ComfyUI"
FRP_DIR = "/content/frp"
FRPC_BIN = f"{FRP_DIR}/frpc"
VPS_IP = "34.153.197.18"

# ==================================================
# 🛒 应用商店开关 (ComfyUI-Manager)
# False = 日常极速跑图 (屏蔽 Manager，极速启动)
# True  = 需要安装新插件时打开 (恢复慢速下载缓存)
# ==================================================
ENABLE_MANAGER = False

print("🛑 1. 秒杀旧进程，释放端口...")
os.system("pkill -f 'main.py'")
os.system("pkill -f frpc")
time.sleep(1)

print(f"🎛️ 2. 处理 Manager 状态 (当前: {'开启' if ENABLE_MANAGER else '极致屏蔽'})...")
MANAGER_DIR = f"{COMFY_DIR}/custom_nodes/ComfyUI-Manager"
HIDDEN_MANAGER_DIR = f"{COMFY_DIR}/ComfyUI-Manager_hidden"

if ENABLE_MANAGER:
    if os.path.exists(HIDDEN_MANAGER_DIR):
        os.rename(HIDDEN_MANAGER_DIR, MANAGER_DIR)
else:
    if os.path.exists(MANAGER_DIR):
        os.rename(MANAGER_DIR, HIDDEN_MANAGER_DIR)

print("🚀 3. 极速启动 ComfyUI 主引擎...")
# Colab 免费版分配的通常是 T4，保留 normalvram 防爆显存
comfy = subprocess.Popen(["python3", "main.py", "--normalvram"], cwd=COMFY_DIR)

print("⏳ 4. 等待引擎点火...")
for _ in range(60):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", 8188)) == 0:
            break
    time.sleep(1)

print("🌐 5. 端口就绪！唤醒 FRP 隧道...")
frpc = subprocess.Popen([FRPC_BIN, "-c", f"{FRP_DIR}/frpc.toml"])

print("\n" + "="*50)
print(f"🎉 Colab 引擎已极速拉起！(应用商店: {'已加载' if ENABLE_MANAGER else '已隐藏'})")
print(f"👉 请刷新网页: http://{VPS_IP}:8188")
print("="*50 + "\n")

comfy.wait()

🛑 1. 秒杀旧进程，释放端口...
🎛️ 2. 处理 Manager 状态 (当前: 极致屏蔽)...
🚀 3. 极速启动 ComfyUI 主引擎...
⏳ 4. 等待引擎点火...
🌐 5. 端口就绪！唤醒 FRP 隧道...

🎉 Colab 引擎已极速拉起！(应用商店: 已隐藏)
👉 请刷新网页: http://34.153.197.18:8188



KeyboardInterrupt: 